# Download Yellow Taxi Trip Data to ADLS

Downloads `yellow_tripdata_2020-01.parquet` from the NYC TLC public dataset and writes it to Azure Data Lake Storage Gen2 using the workspace-configured credentials.

In [ ]:
SOURCE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2020-01.parquet"
FILE_NAME  = "yellow_tripdata_2020-01.parquet"
LOCAL_TMP  = f"/tmp/{FILE_NAME}"
DEST_PATH  = f"abfss://temp@wbcdaddatalakesa.dfs.core.windows.net/kent/inbound/{FILE_NAME}"

In [ ]:
import requests

print(f"Downloading from: {SOURCE_URL}")

response = requests.get(SOURCE_URL, stream=True, timeout=120)
response.raise_for_status()

with open(LOCAL_TMP, "wb") as f:
    for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
        f.write(chunk)

import os
size_mb = os.path.getsize(LOCAL_TMP) / 1024 / 1024
print(f"Saved to {LOCAL_TMP} ({size_mb:.1f} MB)")

In [ ]:
print(f"Copying to: {DEST_PATH}")

dbutils.fs.cp(f"file://{LOCAL_TMP}", DEST_PATH)

print("Upload complete.")

In [ ]:
# Verify
info = dbutils.fs.ls("abfss://temp@wbcdaddatalakesa.dfs.core.windows.net/kent/inbound/")
for f in info:
    print(f"{f.name}  {f.size / 1024 / 1024:.1f} MB")